In [ ]:
import pandas as pd
import numpy as np
import mlflow.sklearn

import dagshub
dagshub.init(repo_owner="AndriaMakharadze", repo_name="IEEE_Fraud_Detection_AM", mlflow=True)

In [18]:
model = mlflow.sklearn.load_model("models:/XGBoost_FraudDetection/1")
print("Model loaded successfully!")

Model loaded successfully!


In [19]:
test_transaction = pd.read_csv("../data/test_transaction.csv")
test_identity = pd.read_csv("../data/test_identity.csv")

test = test_transaction.merge(test_identity, on="TransactionID", how="left")
print("Test shape:", test.shape)

Test shape: (506691, 433)


In [20]:
transaction_ids = test["TransactionID"]
X_test = test.drop(columns=["TransactionID"])

# Feature engineering
X_test["TransactionAmt_log"] = np.log1p(X_test["TransactionAmt"])
X_test["hour_of_day"] = (X_test["TransactionDT"] / 3600).astype(int) % 24
X_test["null_count"] = X_test.isnull().sum(axis=1)

# Add any missing columns that were in training
missing_cols = ['id_01', 'id_02', 'id_05', 'id_06', 'id_11', 'id_12', 'id_13', 
                'id_15', 'id_16', 'id_19', 'id_20', 'id_28', 'id_29', 'id_31', 
                'id_35', 'id_36', 'id_37', 'id_38']
for col in missing_cols:
    if col not in X_test.columns:
        X_test[col] = np.nan

print("Preprocessing done. Shape:", X_test.shape)

Preprocessing done. Shape: (506691, 453)


In [21]:
preds = model.predict_proba(X_test)[:, 1]
print("Predictions done!")

Predictions done!


In [22]:
submission = pd.DataFrame({
    "TransactionID": transaction_ids,
    "isFraud": preds
})

submission.to_csv("../data/submission.csv", index=False)
print("submission.csv saved!")
print(submission.head())

submission.csv saved!
   TransactionID   isFraud
0        3663549  0.004289
1        3663550  0.011681
2        3663551  0.018266
3        3663552  0.010656
4        3663553  0.014194
